In [2]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd 
import scipy as sp 
import scipy.stats as sps
import astropy
from astropy.io import fits
from scipy.stats import poisson
import statsmodels as sm
from scipy.special import gammaln
from matplotlib.ticker import MultipleLocator, AutoMinorLocator


# colors 
# special plot params
plt.style.use('dark_background')
plt.rcParams['axes.facecolor'] = 'midnightblue'
plt.rcParams['figure.facecolor'] = "#060552"
plt.rcParams['axes.grid'] = 'True'
plt.rcParams['grid.color'] = 'steelblue'
plt.rcParams['grid.linewidth'] = 0.6
plt.rcParams['grid.alpha'] = 0.8
plt.rcParams['axes.edgecolor'] = 'white'
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['legend.facecolor'] = "#060552"



### First pass at analysis of data from October 5, 2026

In [23]:
# name variables, run glob to get files

directory_path = '/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/our_data051026/full/'
files_glob = glob.glob(str(directory_path) + '*.txt')
files_glob

['/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/our_data051026/full/HESSJ0632+057_1185418.txt',
 '/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/our_data051026/full/HESSJ0632+057_1185417.txt',
 '/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/our_data051026/full/HESSJ0632+057_1185416.txt']

In [27]:
flux_arrays = []
wavelength_arrays = []

for file in files_glob: 
    spectrum = np.loadtxt(file)
    spectrum_flux = spectrum[:,1]
    spectrum_wavelength = spectrum[:,0]
    flux_arrays.append(spectrum_flux)
    wavelength_arrays.append(spectrum_wavelength)


In [29]:
# # check that they're the same or not 

print(wavelength_arrays[0] == wavelength_arrays[1])
# print(wavelength_arrays[1] == wavelength_arrays[2])

[ True  True  True ...  True  True  True]


1185418 and 1185417 have the same length ^ 

In [31]:
print(len(wavelength_arrays[0]), len(wavelength_arrays[1]), len(wavelength_arrays[2]))

167306 167306 167307


Comparing length with NaN removal and not

In [56]:
def import_txts_get_arrays(folder_in_data_dir):  # this argument is a string
    '''
    This function: takes files from .txt in the data directory
    that holds the files that were output from the fits_to_txt_v3.py
    and takes those files, loads them in as numpy arrays, and then 
    creates an output of 2D wavelength arrays and flux arrays 
    
    input: 
    - exact folder where the data was originally (the "full" is included in the function) 
    output: 
    - 2 2D arrays 

    need to change per person: 
    - data_dir_path: wherever your general data folder is living 
    '''


    data_dir_path = '/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/' 
    files_glob = glob.glob(str(data_dir_path) + folder_in_data_dir + 'full/*.txt')
    wavelength_arrays = []
    flux_arrays = []
    spectra = []
    for file in files_glob: 
        spectrum = np.loadtxt(file)
        spectrum_wavelength = spectrum[:,0]
        spectrum_flux = spectrum[:,1]
        wavelength_arrays.append(spectrum_wavelength)
        flux_arrays.append(spectrum_flux)
        spectra.append(spectrum)
    return wavelength_arrays, flux_arrays, spectra


In [59]:
wave_no_nans, flux_no_nans, spectra_no_nans = import_txts_get_arrays(folder_in_data_dir = 'our_data051026_noNaNremoval/')

In [ ]:
def check_length_all_spectra(wavelength_array): 
    for i in range(len(wave_no_nans)): 
        wavelength_array = wave_no_nans[i]
        print(len(wavelength_array))

In [40]:
for i in range(len(wave_no_nans)): 
    wavelength_array = wave_no_nans[i]
    print(len(wavelength_array))

167306
167306
167307


In [41]:
print(min(wavelength_arrays[0]), max(wavelength_arrays[0]))
print(min(wavelength_arrays[1]), max(wavelength_arrays[1]))
print(min(wavelength_arrays[2]), max(wavelength_arrays[2]))

3773.12406 9006.74602
3773.12406 9006.74602
3773.12406 9006.79286


In [47]:
np.log(9006.79286 - 9006.74602)

np.float64(-3.061017740272578)

This ^ is the extra wavelength channel that was added to the last spectra for some reason. 

In [ ]:
wavelength_width = np.diff(wavelength_arrays[0])

In [45]:
wavelength_width_log = np.diff(np.log(wavelength_arrays[0]))

In [44]:
wavelength_width

array([0.01962, 0.01963, 0.01962, ..., 0.04684, 0.04684, 0.04684],
      shape=(167305,))

In [46]:
wavelength_width_log

array([5.19992152e-06, 5.20254478e-06, 5.19986743e-06, ...,
       5.20061395e-06, 5.20058690e-06, 5.20055986e-06], shape=(167305,))

^ because we literally construct our own wavelength axis in log scale, then the np.diff without log made no sense, because it was in log. which is why when you do np.diff(np.log()) they are all the same step size! 

### Interpolation

In [74]:
# define wavelength grid by the first spectrum as reference 

wl_ref = spectra_no_nans[0][:, 0] # reference wavelength array
fluxes_aligned = [] # make empty list for all the interpolated fluxes 
for i in range(len(spectra_no_nans)): # iterate through the spectra 
    wl = spectra_no_nans[i][:, 0] 
    flux = spectra_no_nans[i][:, 1] 
    # interpolate on the reference axix
    aligned_flux = np.interp(x = wl_ref, 
                             xp = wl, 
                             fp = flux)
    # append to array for interpolated fluxes                            
    fluxes_aligned.append(aligned_flux)

In [75]:
fluxes_aligned

[array([  -2.51,   -3.82,   -2.74, ...,  980.77, 1028.43, 1126.06],
       shape=(167306,)),
 array([  -4.56,   -4.82,   -4.13, ...,  949.36,  958.7 , 1084.46],
       shape=(167306,)),
 array([-9.6000e-01, -3.2700e+00, -4.9300e+00, ...,  8.7076e+02,
         9.1575e+02,  9.6746e+02], shape=(167306,))]

#### Cool! ^ Now they are all aligned (interpolated) on one grid. 

Now going to get the median value between the three of them

In [78]:
median_fluxes_interp = np.median(fluxes_aligned, axis=0)
# use np.column_stack 

median_output_file = '/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/our_data051026/full_median/median_combined_051026.txt'

np.savetxt(median_output_file, 
           np.column_stack((wl_ref, median_fluxes_interp)), 
           fmt="%.8f %.10e")



In [79]:
# just doing this so i can try to normalize again: 

median_fluxes_interp = np.median(fluxes_aligned, axis=0)
# use np.column_stack 

median_output_file = '/Users/leayamashiro/AnA_MSc/Observations/LaPalma_analysis/data/our_data051026/full_median/median_combined_2_051026.txt'

np.savetxt(median_output_file, 
           np.column_stack((wl_ref, median_fluxes_interp)), 
           fmt="%.8f %.10e")

